In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


In [ ]:
SEED = 42
OUT  = '../output/'
RAW  = r'C:\Users\DorothyCheruiyot\Desktop\Projects\insurance_claims\data\insurance_data_features.xlsx'


In [3]:
# metrics
def gamma_deviance(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 1e-10, None)
    return float(2 * np.sum(-np.log(y_true / y_pred) + (y_true - y_pred) / y_pred))

def calibration_pct(y_true, y_pred):
    return float((np.sum(y_pred) - np.sum(y_true)) / np.sum(y_true) * 100)

def gini_coefficient(y_true, y_pred):
    idx = np.argsort(y_pred)
    ys  = np.array(y_true, dtype=float)[idx]
    cs  = np.cumsum(ys)
    if cs[-1] == 0: return 0.0
    return float((len(ys) + 1 - 2 * np.sum(cs) / cs[-1]) / len(ys))

def lift_table(y_true, y_pred, n=10):
    df_l = pd.DataFrame({'y': np.array(y_true, float), 'p': np.array(y_pred, float)})
    df_l['d'] = pd.qcut(df_l['p'], n, labels=False, duplicates='drop')
    return df_l.groupby('d').agg(actual=('y','mean'), predicted=('p','mean'),
                                  count=('y','count')).reset_index()

def score(name, y_true, y_pred):
    y_t = np.array(y_true, dtype=float)
    y_p = np.array(y_pred, dtype=float)
    return {
        'model'           : name,
        'gamma_deviance'  : round(gamma_deviance(y_t, y_p), 4),
        'calibration_%'   : round(calibration_pct(y_t, y_p), 3),
        'gini'            : round(gini_coefficient(y_t, y_p), 4),
        'mae'             : round(float(mean_absolute_error(y_t, y_p)), 2),
        'rmse'            : round(float(np.sqrt(np.mean((y_t-y_p)**2))), 2),
        'mean_actual'     : round(float(np.mean(y_t)), 2),
        'mean_predicted'  : round(float(np.mean(y_p)), 2),
        'total_actual'    : round(float(np.sum(y_t)), 0),
        'total_predicted' : round(float(np.sum(y_p)), 0),
    }

METRIC_KEYS = ['gamma_deviance','calibration_%','gini','mae','rmse',
               'mean_actual','mean_predicted','total_actual','total_predicted']
METRIC_LBLS = ['Gamma Deviance','Calibration %','Gini',
               'MAE (£)','RMSE (£)','Mean Actual (£)','Mean Predicted (£)',
               'Total Actual (£)','Total Predicted (£)']

In [15]:
sheets    = pd.read_excel(RAW, sheet_name=None, dtype=str)
customers = sheets['Customers'].copy()
policies  = sheets['Policies'].copy()
claims    = sheets['Claims'].copy()
external  = sheets['External'].copy()

claims['Fraud Flag']= pd.to_numeric(claims['Fraud Flag'],  errors='coerce')
claims['Claim Amount']= pd.to_numeric(claims['Claim Amount'],errors='coerce')
claims['Settlement Days'] = pd.to_numeric(claims['Settlement Days'], errors='coerce')
claims['Claim Date']= pd.to_datetime(claims['Claim Date'], errors='coerce')
claims['Settlement Date'] = pd.to_datetime(claims['Settlement Date'],errors='coerce')
claims['year_month'] = claims['Claim Date'].dt.to_period('M').astype(str)

for c in ['Annual Premium','Excess Amount','Sum Insured','Renewal Count']:
    policies[c] = pd.to_numeric(policies[c], errors='coerce')

policies['Start Date'] = pd.to_datetime(policies['Start Date'], errors='coerce')
policies['End Date']= pd.to_datetime(policies['End Date'],   errors='coerce')
policies['policy_duration_days'] = (policies['End Date'] - policies['Start Date']).dt.days.clip(lower=1)

for c in ['Age','Credit Score','Prior Claims Count','Tenure Years','Late Payments 12M']:
    customers[c] = pd.to_numeric(customers[c], errors='coerce')

for c in ['Avg Rainfall Mm','Avg Wind Speed Kmh','Flood Risk Index',
          'Gdp Growth Rate','Cpi Inflation','Unemployment Rate',
          'Avg Property Price Gbp','Storm Event Flag']:
    external[c] = pd.to_numeric(external[c], errors='coerce')

abt = claims.merge(
    policies[['Policy Id','Customer Id','Coverage Type','Annual Premium',
              'Excess Amount','Sum Insured','Renewal Count',
              'Distribution Channel','Start Date','End Date',
              'policy_duration_days']],
    on='Policy Id', how='left', suffixes=('_cl','_pol'))
abt = abt.rename(columns={'Customer Id_cl':'Customer Id',
                           'Coverage Type_cl':'Coverage Type'})
abt = abt.merge(
    customers[['Customer Id','Age','Gender','Region','Income Band','Credit Score',
               'Prior Claims Count','Tenure Years','Payment Pattern',
               'Late Payments 12M','Occupation']],
    on='Customer Id', how='left')

ext = external.rename(columns={'Year Month':'year_month'})
abt = abt.merge(
    ext[['Region','year_month','Avg Rainfall Mm','Avg Wind Speed Kmh',
         'Flood Risk Index','Gdp Growth Rate','Cpi Inflation',
         'Unemployment Rate','Avg Property Price Gbp','Storm Event Flag']],
    on=['Region','year_month'], how='left')

sev = abt[(abt['Claim Status']=='Settled') & (abt['Fraud Flag']==0)].copy()
sev = sev.reset_index(drop=True)
sev.head()

,Claim Id,Policy Id,Customer Id,Claim Date,Claim Type,Claim Amount,Claim Status,Settlement Days,Settlement Date,Fraud Flag,...,Late Payments 12M,Occupation,Avg Rainfall Mm,Avg Wind Speed Kmh,Flood Risk Index,Gdp Growth Rate,Cpi Inflation,Unemployment Rate,Avg Property Price Gbp,Storm Event Flag
0,CLM0000001,POL001589,CUST04742,2021-02-28,Accident,3070.31,Settled,118,2021-06-26,0,...,0,Diplomatic Services operational officer,30.4,11.5,1.83,0.0115,0.0631,0.0539,328500,0
1,CLM0000002,POL010598,CUST09209,2024-07-18,Medical,2668.14,Settled,83,2024-10-09,0,...,0,Data scientist,78.5,20.5,3.55,0.0113,0.0406,0.0443,280400,0
2,CLM0000003,POL002080,CUST01646,2022-11-16,Accident,19384.22,Settled,116,2023-03-12,0,...,0,Logistics and distribution manager,17.4,13.0,1.00,0.0099,0.0385,0.0345,219700,1
3,CLM0000004,POL006528,CUST09866,2021-06-13,Theft,3965.53,Settled,13,2021-06-26,0,...,1,"Programme researcher, broadcasting/film/video",63.0,16.6,3.02,0.0195,0.0328,0.0374,357000,0
4,CLM0000008,POL003086,CUST09637,2023-07-18,Medical,3786.58,Settled,83,2023-10-09,0,...,0,"Therapist, drama",91.7,20.0,5.00,0.0126,0.0307,0.0284,335000,0


In [ ]:
# feature engineering
income_map = {'<20k':1,'20-40k':2,'40-60k':3,'60-100k':4,'>100k':5}
excess_map = {100:1,150:2,200:3,250:4,300:5,500:6}
sev['income_ord']  = sev['Income Band'].map(income_map)
sev['excess_ord']  = sev['Excess Amount'].map(excess_map)
sev['log_sum_insured']= np.log1p(sev['Sum Insured'])
sev['log_annual_premium'] = np.log1p(sev['Annual Premium'])
sev['log_property_price'] = np.log1p(sev['Avg Property Price Gbp'])
sev['claim_month']= sev['Claim Date'].dt.month
sev['is_winter']  = sev['claim_month'].isin([10,11,12,1,2,3]).astype(int)
sev['excess_to_si'] = sev['Excess Amount'] / sev['Sum Insured'].clip(lower=1)
sev['log_net_exposure'] = np.log1p((sev['Sum Insured'] - sev['Excess Amount']).clip(lower=0))
cs_min = sev['Credit Score'].min(); cs_max = sev['Credit Score'].max()
sev['credit_risk_norm']  = 1 - (sev['Credit Score'] - cs_min) / (cs_max - cs_min + 1e-9)
sev['income_x_credit'] = sev['income_ord'] * sev['Credit Score'] / 1000
sev['late_payment_rate'] = sev['Late Payments 12M'] / (sev['Tenure Years'] + 1)
sev['settlement_bucket'] = pd.cut(sev['Settlement Days'],bins=[0,30,90,150,999], labels=[1,2,3,4]).astype(float)
sev['settlement_days_sq'] = sev['Settlement Days'] ** 2
sev['log_settlement_days'] = np.log1p(sev['Settlement Days'])
sev['loyalty_score'] = sev['Renewal Count'] * sev['Tenure Years']


sev['policy_age_at_claim'] = (sev['Claim Date'] - sev['Start Date']).dt.days.clip(lower=0)
sev['log_policy_age']      = np.log1p(sev['policy_age_at_claim'])
sev['early_claim_flag']    = (sev['policy_age_at_claim'] <= 60).astype(int)
sev['days_to_expiry']      = (sev['End Date'] - sev['Claim Date']).dt.days.clip(lower=0)
sev['near_expiry_flag']    = (sev['days_to_expiry'] <= 30).astype(int)
sev['si_per_policy_year']  = sev['Sum Insured'] / (sev['policy_duration_days'] / 365.25).clip(lower=0.1)
sev['log_si_per_year']     = np.log1p(sev['si_per_policy_year'])


rebuild_types = {'Fire', 'Flood', 'Natural Disaster'}
sev['cpi_x_rebuild'] = sev['Cpi Inflation'] * sev['Claim Type'].isin(rebuild_types).astype(float)
sev['age_x_medical'] = sev['Age'] * (sev['Claim Type'] == 'Medical').astype(int)
sev['age_x_accident']= sev['Age'] * (sev['Claim Type'] == 'Accident').astype(int)
sev['prior_claims_x_log_si'] = sev['Prior Claims Count'] * sev['log_sum_insured']

cov_dummies = pd.get_dummies(sev['Coverage Type'], prefix='cov', drop_first=True, dtype=int)
ct_dummies  = pd.get_dummies(sev['Claim Type'],  prefix='ct',drop_first=True, dtype=int)
reg_dummies = pd.get_dummies(sev['Region'],prefix='reg', drop_first=True, dtype=int)
sev = pd.concat([sev, cov_dummies, ct_dummies, reg_dummies], axis=1)

COV_DUMMIES = list(cov_dummies.columns)
CT_DUMMIES  = list(ct_dummies.columns)
REG_DUMMIES = list(reg_dummies.columns)

sev.shape

(2856, 87)

In [ ]:
# splitting
df_trainval, df_test = train_test_split(sev, test_size=0.10, random_state=SEED)
df_train, df_val= train_test_split(df_trainval, test_size=0.1111, random_state=SEED)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test= df_test.reset_index(drop=True)

TARGET = 'Claim Amount'

ct_means_train = df_train.groupby('Claim Type')[TARGET].mean().to_dict()
for df_split in [df_train, df_val, df_test]:
    df_split['excess_coverage_ratio'] = df_split.apply(
        lambda r: r['Excess Amount'] / max(ct_means_train.get(r['Claim Type'], 10000), 1),
        axis=1)

reg_sev_mean = df_train.groupby('Region')[TARGET].mean()
for df_split in [df_train, df_val, df_test]:
    df_split['region_sev_te'] = df_split['Region'].map(reg_sev_mean).fillna(
        df_train[TARGET].mean())

  Train:2,284  Val:286  Test:286
  Mean Claim Amount — Train:£11,182  Val:£10,364  Test:£10,858


In [19]:

SCALE_COLS = [
    'log_sum_insured','Settlement Days','excess_ord','excess_to_si',
    'log_net_exposure','income_x_credit','late_payment_rate',
    'settlement_bucket','settlement_days_sq','log_settlement_days','loyalty_score',
    'log_policy_age','days_to_expiry','log_si_per_year',
    'excess_coverage_ratio','cpi_x_rebuild',
    'age_x_medical','age_x_accident','prior_claims_x_log_si',
    'region_sev_te','log_annual_premium','log_property_price',
    'Age','Credit Score','Prior Claims Count','Tenure Years',
]
SCALE_COLS = [c for c in SCALE_COLS if c in df_train.columns]

scaler = StandardScaler()
scaler.fit(df_train[SCALE_COLS])

def apply_scaler(df_split):
    vals = scaler.transform(df_split[SCALE_COLS])
    for i, c in enumerate(SCALE_COLS):
        df_split[f'{c}_sc'] = vals[:, i]
    return df_split

df_train = apply_scaler(df_train)
df_val   = apply_scaler(df_val)
df_test  = apply_scaler(df_test)


In [ ]:

FEAT_S4 = list(dict.fromkeys([f for f in
    CT_DUMMIES + COV_DUMMIES
    + ['log_sum_insured_sc','Settlement Days_sc','excess_ord_sc',
       'is_winter','Storm Event Flag',
       'excess_to_si_sc','log_net_exposure_sc',
       'income_x_credit_sc','late_payment_rate_sc',
       'settlement_bucket_sc','settlement_days_sq_sc',
       'log_settlement_days_sc','loyalty_score_sc']
    if f in df_train.columns]))

len(FEAT_S4)

24

In [ ]:
def fit_gamma(feats):
    Xc  = sm.add_constant(df_train[feats].fillna(0), has_constant='add')
    glm = sm.GLM(df_train[TARGET], Xc,
                 family=sm.families.Gamma(link=sm.families.links.Log()))
    return glm.fit(maxiter=300, disp=False)

def predict_glm(res, df_split, feats):
    Xc = sm.add_constant(df_split[feats].fillna(0), has_constant='add')
    for c in set(res.params.index) - set(Xc.columns):
        Xc[c] = 0
    return res.predict(np.array(Xc[res.params.index]))

s4_result = fit_gamma(FEAT_S4)
s4_pred_v = predict_glm(s4_result, df_val, FEAT_S4)
BASE_DEV  = gamma_deviance(df_val[TARGET], s4_pred_v)
print(f'  S4 baseline val deviance: {BASE_DEV:.4f}\n')

NEW_CANDIDATES = [
    ('log_policy_age_sc','log(days since policy start at claim date)'),
    ('early_claim_flag','Claim in first 60 days — moral hazard flag'),
    ('days_to_expiry_sc','Days remaining to policy expiry at claim'),
    ('near_expiry_flag','Claim within 30 days of policy expiry'),
    ('log_si_per_year_sc','log(Sum Insured / policy duration years)'),
    ('excess_coverage_ratio_sc', 'Excess / typical claim amount for claim type'),
    ('cpi_x_rebuild_sc', 'CPI and rebuild claim type (Fire/Flood/NatDis)'),
    ('age_x_medical_sc', 'Age and Medical claim type interaction'),
    ('age_x_accident_sc','Age and Accident claim type interaction'),
    ('prior_claims_x_log_si_sc','Prior claim count and log(Sum Insured)'),
    ('region_sev_te_sc',  'Region target-encoded vs mean severity (train)'),
] + [(f, f'Region dummy: {f}') for f in REG_DUMMIES if f in df_train.columns] 

NEW_CANDIDATES = [(f, d) for f, d in NEW_CANDIDATES if f in df_train.columns]

current_feats = list(FEAT_S4)
current_dev   = BASE_DEV
stepwise_log  = []

for i, (feat, desc) in enumerate(NEW_CANDIDATES, 1):
    if feat in current_feats:
        continue
    trial_feats= current_feats + [feat]
    trial_result = fit_gamma(trial_feats)
    trial_pred = predict_glm(trial_result, df_val, trial_feats)
    trial_dev= gamma_deviance(df_val[TARGET], trial_pred)
    trial_mae= mean_absolute_error(df_val[TARGET], trial_pred)
    delta = trial_dev - current_dev
    kept  = trial_dev < current_dev

    stepwise_log.append({'feature': feat, 'description': desc,
                         'trial_dev': round(trial_dev,4), 'prev_dev': round(current_dev,4),
                         'delta': round(delta,4), 'trial_mae': round(trial_mae,2), 'kept': kept})
    if kept:
        current_feats = trial_feats
        current_dev   = trial_dev

FEAT_S5  = current_feats
kept_new = [r['feature'] for r in stepwise_log if r['kept']]

pd.DataFrame(stepwise_log).to_csv(f'{OUT}sev_final_stepwise_log.csv', index=False)


  S4 baseline val deviance: 70.8868



In [10]:
s5_result = fit_gamma(FEAT_S5)
s5_pred_v = predict_glm(s5_result, df_val,  FEAT_S5)
s5_pred_t = predict_glm(s5_result, df_test, FEAT_S5)
s4_pred_t = predict_glm(s4_result, df_test, FEAT_S4)

print(f'  S5 AIC      : {s5_result.aic:.1f}')
print(f'  Pseudo R²   : {1 - s5_result.deviance/s5_result.null_deviance:.4f}')
print(f'  Features    : {len(FEAT_S5)}')

  S5 AIC      : 43753.5
  Pseudo R²   : 0.7527
  Features    : 33


In [ ]:
# tree-based models
FEAT_TREE = list(dict.fromkeys([f for f in
    CT_DUMMIES + COV_DUMMIES + REG_DUMMIES 
    + ['Settlement Days','log_sum_insured','excess_ord','is_winter',
       'Storm Event Flag','excess_to_si','log_net_exposure',
       'income_x_credit','late_payment_rate','settlement_bucket',
       'settlement_days_sq','log_settlement_days','loyalty_score',
       'log_policy_age','early_claim_flag','days_to_expiry',
       'near_expiry_flag','log_si_per_year','excess_coverage_ratio',
       'cpi_x_rebuild','age_x_medical','age_x_accident',
       'prior_claims_x_log_si','region_sev_te',
       'log_annual_premium','log_property_price',
       'Age','Credit Score','Prior Claims Count','Tenure Years',
       'Late Payments 12M','Renewal Count',
       'Avg Rainfall Mm','Flood Risk Index','Cpi Inflation',
       'Unemployment Rate']
    if f in df_train.columns]))

X_tr = df_train[FEAT_TREE].fillna(0)
X_vl = df_val[FEAT_TREE].fillna(0)
X_ts = df_test[FEAT_TREE].fillna(0)
y_tr = df_train[TARGET]; y_vl = df_val[TARGET]; y_ts = df_test[TARGET]

# XGBoost
print('  Fitting XGBoost (reg:gamma)...')
xgb_model = xgb.XGBRegressor(
    objective='reg:gamma', n_estimators=500, learning_rate=0.03,
    max_depth=4, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=10, gamma=2, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbosity=0)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
xgb_pred_v = np.clip(xgb_model.predict(X_vl), 1, None)
xgb_pred_t = np.clip(xgb_model.predict(X_ts), 1, None)
print(f'    Val  MAE=£{mean_absolute_error(y_vl, xgb_pred_v):,.0f}  Dev={gamma_deviance(y_vl, xgb_pred_v):.4f}')

# LightGBM
print('  Fitting LightGBM (gamma objective)...')
lgb_model = lgb.LGBMRegressor(
    objective='gamma', n_estimators=500, learning_rate=0.03,
    max_depth=4, num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1)
lgb_model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
lgb_pred_v = np.clip(lgb_model.predict(X_vl), 1, None)
lgb_pred_t = np.clip(lgb_model.predict(X_ts), 1, None)
print(f'    Val  MAE=£{mean_absolute_error(y_vl, lgb_pred_v):,.0f}  Dev={gamma_deviance(y_vl, lgb_pred_v):.4f}')

# Random Forest
print('  Fitting Random Forest...')
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=8, min_samples_leaf=10,
    max_features='sqrt', random_state=SEED, n_jobs=-1)
rf_model.fit(X_tr, y_tr)
rf_pred_v = np.clip(rf_model.predict(X_vl), 1, None)
rf_pred_t = np.clip(rf_model.predict(X_ts), 1, None)
print(f'    Val  MAE=£{mean_absolute_error(y_vl, rf_pred_v):,.0f}  Dev={gamma_deviance(y_vl, rf_pred_v):.4f}')

# Tweedie
print('  Fitting Tweedie (power=1.5)...')
FEAT_SCALED_ALL = [f'{c}_sc' for c in SCALE_COLS if f'{c}_sc' in df_train.columns]
FEAT_TWEEDIE = list(dict.fromkeys([f for f in
    CT_DUMMIES + COV_DUMMIES + REG_DUMMIES
    + FEAT_SCALED_ALL + ['is_winter','Storm Event Flag','early_claim_flag','near_expiry_flag']
    if f in df_train.columns]))
tw_model = TweedieRegressor(power=1.5, alpha=0.01, max_iter=500, link='log')
tw_model.fit(df_train[FEAT_TWEEDIE].fillna(0), y_tr)
tw_pred_v = np.clip(tw_model.predict(df_val[FEAT_TWEEDIE].fillna(0)),  1, None)
tw_pred_t = np.clip(tw_model.predict(df_test[FEAT_TWEEDIE].fillna(0)), 1, None)
print(f'    Val  MAE=£{mean_absolute_error(y_vl, tw_pred_v):,.0f}  Dev={gamma_deviance(y_vl, tw_pred_v):.4f}')

  Fitting XGBoost (reg:gamma)...
    Val  MAE=£4,189  Dev=74.0521
  Fitting LightGBM (gamma objective)...
    Val  MAE=£4,149  Dev=73.1324
  Fitting Random Forest...
    Val  MAE=£4,368  Dev=82.5398
  Fitting Tweedie (power=1.5)...
    Val  MAE=£4,196  Dev=71.4955


In [22]:
grand_mean = df_train[TARGET].mean()
b1_pred_v  = np.full(len(df_val),  grand_mean)
b1_pred_t  = np.full(len(df_test), grand_mean)
grand_mean

np.float64(11182.488721541156)

In [ ]:
# score all models
MODEL_DEFS = [
    ('B1 Grand Mean', b1_pred_v,  b1_pred_t),
    ('S4 Gamma GLM',  s4_pred_v,  s4_pred_t),
    ('S5 Gamma GLM',  s5_pred_v,  s5_pred_t),
    ('XGBoost', xgb_pred_v, xgb_pred_t),
    ('LightGBM', lgb_pred_v, lgb_pred_t),
    ('Random Forest', rf_pred_v,  rf_pred_t),
    ('Tweedie', tw_pred_v,  tw_pred_t),
]
model_names = [n for n, _, _ in MODEL_DEFS]
val_scores  = {n: score(n, y_vl, pv) for n, pv, _  in MODEL_DEFS}
test_scores = {n: score(n, y_ts, pt) for n, _,  pt in MODEL_DEFS}

# Display as DataFrame
val_df  = pd.DataFrame(val_scores).T[METRIC_KEYS]
test_df = pd.DataFrame(test_scores).T[METRIC_KEYS]
val_df.columns  = METRIC_LBLS
test_df.columns = METRIC_LBLS

print('VALIDATION SET')
display(val_df.style.highlight_min(axis=0, subset=METRIC_LBLS[:5], color='#d4edda')
               .highlight_max(axis=0, subset=['Gini'], color='#d4edda'))

print('\nTEST SET')
display(test_df.style.highlight_min(axis=0, subset=METRIC_LBLS[:5], color='#d4edda')
                .highlight_max(axis=0, subset=['Gini'], color='#d4edda'))

rows = []
for split_n, scores_d in [('val', val_scores), ('test', test_scores)]:
    for mn in model_names:
        for key in METRIC_KEYS:
            rows.append({'split':split_n,'model':mn,'metric':key,'value':scores_d[mn][key]})
pd.DataFrame(rows).to_csv(f'{OUT}sev_final_comparison.csv', index=False)


VALIDATION SET


,Gamma Deviance,Calibration %,Gini,MAE (£),RMSE (£),Mean Actual (£),Mean Predicted (£),Total Actual (£),Total Predicted (£)
B1 Grand Mean,282.211100,7.892000,-0.025000,8534.880000,10905.580000,10364.470000,11182.490000,2964239.000000,3198192.000000
S4 Gamma GLM,70.886800,5.997000,0.443400,4170.970000,6824.260000,10364.470000,10985.980000,2964239.000000,3141990.000000
S5 Gamma GLM,69.778900,5.778000,0.446600,4090.290000,6666.570000,10364.470000,10963.330000,2964239.000000,3135513.000000
XGBoost,74.052100,3.010000,0.436200,4188.500000,6897.150000,10364.470000,10676.410000,2964239.000000,3053452.000000
LightGBM,73.132400,0.893000,0.438600,4149.430000,6924.180000,10364.470000,10457.060000,2964239.000000,2990720.000000
Random Forest,82.539800,5.646000,0.434200,4367.740000,6999.010000,10364.470000,10949.640000,2964239.000000,3131598.000000
Tweedie,71.495500,5.593000,0.442300,4195.870000,6813.320000,10364.470000,10944.110000,2964239.000000,3130014.000000



TEST SET


,Gamma Deviance,Calibration %,Gini,MAE (£),RMSE (£),Mean Actual (£),Mean Predicted (£),Total Actual (£),Total Predicted (£)
B1 Grand Mean,251.093400,2.988000,-0.024200,8052.280000,11209.880000,10858.060000,11182.490000,3105405.000000,3198192.000000
S4 Gamma GLM,63.294200,5.138000,0.423900,4361.490000,7413.380000,10858.060000,11416.000000,3105405.000000,3264975.000000
S5 Gamma GLM,63.704200,4.720000,0.423100,4363.690000,7430.400000,10858.060000,11370.510000,3105405.000000,3251967.000000
XGBoost,63.045600,1.397000,0.419100,4284.360000,7421.770000,10858.060000,11009.790000,3105405.000000,3148800.000000
LightGBM,66.051000,-0.460000,0.412800,4333.920000,7663.310000,10858.060000,10808.150000,3105405.000000,3091132.000000
Random Forest,66.000400,4.270000,0.417000,4394.880000,7548.870000,10858.060000,11321.740000,3105405.000000,3238019.000000
Tweedie,65.329000,4.066000,0.420800,4393.990000,7554.520000,10858.060000,11299.510000,3105405.000000,3231660.000000


In [23]:
coef_df = pd.DataFrame({
    'feature' : s5_result.params.index,
    'exp_coef': np.exp(s5_result.params.values),
    'p_value' : s5_result.pvalues.values,
}).query("feature != 'const'").sort_values('p_value').reset_index(drop=True)

coef_df.to_csv(f'{OUT}sev_final_coefficients.csv', index=False)

display(coef_df[['feature','exp_coef','p_value']])

# Feature importance
fi_xgb = pd.Series(xgb_model.feature_importances_, index=FEAT_TREE).sort_values(ascending=False).head(15)
fi_lgb = pd.Series(lgb_model.feature_importances_, index=FEAT_TREE).sort_values(ascending=False).head(15)

print('\n  XGBoost top 10:')
for f, v in fi_xgb.head(10).items():
    print(f'    {f:<40} {v:.4f}')
print('\n  LightGBM top 10:')
for f, v in fi_lgb.head(10).items():
    print(f'    {f:<40} {v:.4f}')

,feature,exp_coef,p_value
0,ct_Fire,3.271218,1.874903e-81
1,ct_Medical,0.316709,3.178253e-64
2,ct_Theft,0.466786,1.924243e-60
3,ct_Natural Disaster,2.475744,1.545089e-55
4,ct_Flood,2.076574,3.214319e-31
5,ct_Liability,1.459843,3.993264e-13
6,reg_North East,0.931755,4.796573e-02
7,log_si_per_year_sc,1.080504,1.135602e-01
8,settlement_bucket_sc,1.049945,1.619114e-01
9,reg_South East,1.048397,2.251235e-01



  XGBoost top 10:
    age_x_medical                            0.2777
    excess_coverage_ratio                    0.1712
    ct_Medical                               0.1417
    ct_Theft                                 0.0943
    cpi_x_rebuild                            0.0703
    cov_Motor                                0.0338
    cov_Health                               0.0264
    age_x_accident                           0.0219
    ct_Liability                             0.0191
    excess_ord                               0.0089

  LightGBM top 10:
    excess_coverage_ratio                    107.0000
    cpi_x_rebuild                            91.0000
    age_x_accident                           77.0000
    excess_to_si                             71.0000
    Unemployment Rate                        71.0000
    ct_Fire                                  62.0000
    Cpi Inflation                            57.0000
    Avg Rainfall Mm                          56.0000
    log_property